# Module 09: Consistent Hashing Distributed Partitioning — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/consistent_hash_ring.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import consistent_hash_ring

classes = [n for n, o in inspect.getmembers(consistent_hash_ring, inspect.isclass)
           if o.__module__ == 'consistent_hash_ring']
functions = [n for n, o in inspect.getmembers(consistent_hash_ring, inspect.isfunction)
             if o.__module__ == 'consistent_hash_ring']

print('module   : consistent_hash_ring')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(consistent_hash_ring, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Deterministic key lookup

This is the module's own `test_deterministic_key_lookup` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from consistent_hash_ring import ConsistentHashRing

ring = ConsistentHashRing(nodes=["node-A", "node-B", "node-C"], vnodes=100)

# Identical keys must always resolve to the exact same physical node
node1 = ring.get_node("user_session_99214")
node2 = ring.get_node("user_session_99214")
assert node1 is not None
assert node1 == node2
assert node1 in {"node-A", "node-B", "node-C"}

print('PASSED: test_deterministic_key_lookup')

## 3. 🔮 Prediction — commit before you run

Adding one node to a 3-node ring: predict what fraction of 1,000 keys get remapped. Compare your guess against 1/4 (naive modulo) and against 1/N.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_minimal_key_migration_on_node_addition`, which tests exactly this property.


In [ ]:
initial_nodes = ["node-1", "node-2", "node-3"]
ring = ConsistentHashRing(nodes=initial_nodes, vnodes=150)

keys = [f"cache_key_{i}" for i in range(1000)]
mapping_before = {k: ring.get_node(k) for k in keys}

# Add a 4th node
ring.add_node("node-4")
mapping_after = {k: ring.get_node(k) for k in keys}

# Count how many keys changed their destination node
migrated_keys = [k for k in keys if mapping_before[k] != mapping_after[k]]
migration_ratio = len(migrated_keys) / len(keys)

# In naive modulo hashing, adding 1 node moves ~75% of keys!
# In consistent hashing, ideal migration to node-4 is 1/4 = 25%.
# With 150 vnodes, migration should be tightly bounded between 15% and 35%.
assert 0.15 <= migration_ratio <= 0.35

# Any key that DID migrate must have migrated to the newly added node-4!
for k in migrated_keys:
    assert mapping_after[k] == "node-4"

print('PASSED: test_minimal_key_migration_on_node_addition')

## 4. Measure it: Minimal key migration on node removal

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_minimal_key_migration_on_node_removal` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

ring = ConsistentHashRing(nodes=["node-1", "node-2", "node-3"], vnodes=150)
keys = [f"item_{i}" for i in range(1000)]
mapping_before = {k: ring.get_node(k) for k in keys}

# Remove node-3
ring.remove_node("node-3")
mapping_after = {k: ring.get_node(k) for k in keys}

# Keys that were on node-1 or node-2 must NOT have moved!
for k in keys:
    if mapping_before[k] != "node-3":
        assert mapping_after[k] == mapping_before[k]
    else:
        # Keys previously on node-3 must now be reassigned to node-1 or node-2
        assert mapping_after[k] in {"node-1", "node-2"}

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_minimal_key_migration_on_node_removal')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(consistent_hash_ring) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Consistent hashing remaps ~1/N of keys on a node change, not ~all of them.
2. Virtual nodes exist to fix variance, not to fix correctness.
3. A preference list must contain distinct *physical* nodes to survive a failure.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
